In [48]:
import openai, json

client = openai.OpenAI()
messages = []

In [ ]:
from typing import Any
import requests

BASE_URL="https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    """
    Movie API에서 인기 영화 목록을 가져옵니다.
    RETURN: "1. 영화제목 (ID:xxxxx), 2. ..." 형식의 문자열
    """
    try:
        response = requests.get(f"{BASE_URL}/movies", timeout=10)
        response.raise_for_status()
        movies = response.json()

        if not movies:
            return "인기 영화 정보를 찾을 수 없습니다."

        result = ", ".join(
            f"{i+1}. {m['title']} (ID:{m['id']}, 평점:{m['vote_average']})"
            for i, m in enumerate(movies)
        )
        return result

    except requests.exceptions.Timeout:
        return "오류: API 요청 시간이 초과되었습니다."
    except requests.exceptions.HTTPError as e:
        return f"오류: HTTP {e.response.status_code} - {e}"
    except requests.exceptions.RequestException as e:
        return f"오류: 네트워크 요청 실패 - {e}"
    except (KeyError, ValueError) as e:
        return f"오류: 응답 파싱 실패 - {e}"

def get_movie_details(id: int):
    """
    특정 영화의 상세 정보를 가져옵니다.
    :param id: 영화 ID (예: 693134)
    """
    try:
        response = requests.get(f"{BASE_URL}/movies/{id}", timeout=10)
        response.raise_for_status()
        m = response.json()  # ✅ 딕셔너리 하나가 바로 반환됨

        title        = m['title']
        year         = m['release_date'][:4] if m.get('release_date') else '?'
        rating       = m['vote_average']
        overview     = m['overview']
        popularity   = m['popularity']
        poster       = m.get('poster_path', '이미지 없음')

        return (
            f"[{title}] ({year}) | "
            f"평점: {rating} | "
            f"인기도: {popularity} | "
            f"포스터: {poster} | "
            f"줄거리: {overview}"
        )

    except requests.exceptions.Timeout:
        return f"오류: 요청 시간 초과 (ID: {id})"
    except requests.exceptions.HTTPError as e:
        return f"오류: HTTP {e.response.status_code} - 영화 ID {id} 를 찾을 수 없습니다."
    except requests.exceptions.RequestException as e:
        return f"오류: 네트워크 요청 실패 - {e}"
    except (KeyError, ValueError) as e:
        return f"오류: 응답 파싱 실패 - {e}"


def get_similar_movies(id):
    """
    특정 영화와 유사한 영화 목록을 가져옵니다.
    :param id: 기준 영화 ID (예: 693134)
    """
    try:
        response = requests.get(f"{BASE_URL}/movies/{id}/similar", timeout=10)
        response.raise_for_status()
        movies = response.json()

        if not movies:
            return f"ID {id} 와 유사한 영화를 찾을 수 없습니다."

        result = ", ".join(
            f"{i+1}. {m['title']} (ID:{m['id']}, 평점:{m['vote_average']}, 개봉:{m['release_date'][:4]})"
            for i, m in enumerate(movies)
        )
        return result

    except requests.exceptions.Timeout:
        return f"오류: 요청 시간 초과 (ID: {id})"
    except requests.exceptions.HTTPError as e:
        return f"오류: HTTP {e.response.status_code} - 영화 ID {id} 를 찾을 수 없습니다."
    except requests.exceptions.RequestException as e:
        return f"오류: 네트워크 요청 실패 - {e}"
    except (KeyError, ValueError) as e:
        return f"오류: 응답 파싱 실패 - {e}"


""" # --- 실행 테스트 ---
if __name__ == "__main__":
    print(get_popular_movies())
    print("-------------\n")
    print(get_movie_details(693134))
    print("-------------\n")
    print(get_similar_movies(693134))
    print("-------------\n")
 """


FUNCTION_MAP = {
    'get_popular_movies' : get_popular_movies,
    'get_movie_details' : get_movie_details,
    'get_similar_movies' : get_similar_movies,
}

1. Your Heart Will Be Broken (ID:1523145), 2. Avatar: Fire and Ash (ID:83533), 3. Hoppers (ID:1327819), 4. The Super Mario Bros. Movie (ID:502356), 5. The Super Mario Galaxy Movie (ID:1226863), 6. Shelter (ID:1290821), 7. Project Hail Mary (ID:687163), 8. Thrash (ID:1290417), 9. Crime 101 (ID:1171145), 10. Greenland 2: Migration (ID:840464), 11. GOAT (ID:1297842), 12. Mike & Nick & Nick & Alice (ID:1115544), 13. Sniper: No Nation (ID:1641319), 14. The Bride! (ID:1159831), 15. War Machine (ID:1265609), 16. The Mortuary Assistant (ID:1470130), 17. Pretty Lethal (ID:1084187), 18. Humint (ID:1268127), 19. Scream 7 (ID:1159559), 20. The Passion of the Christ (ID:615)
-------------

[Dune: Part Two] (2024) | 평점: 8.131 | 인기도: 22.6709 | 포스터: https://image.tmdb.org/t/p/w780/1pdfLvkbY9ohJlCjQH2CZjjYVvJ.jpg | 줄거리: Follow the mythic journey of Paul Atreides as he unites with Chani and the Fremen while on a path of revenge against the conspirators who destroyed his family. Facing a choice between t

In [39]:
TOOLS = [ 
    {
        "type": "function",
        "function" : {
            "name" : "get_popular_movies",
            "description" : "현재 인기 있는 영화 목록을 가져옵니다. 파라미터가 필요 없습니다.",
            "parameters" : {
                "type": "object", 
                "properties": {},
            },
            "required": []
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 특정 영화의 상세 정보를 조회합니다. (줄거리, 평점, 장르, 런타임 등)",
            "parameters": {
                "type": "object", 
                "properties": {
                    "id" : {
                        "type": "integer",
                        "description": "조회할 영화의 ID (예: 693134)"
                    }, 
                },
            },
            "required": ["id"]
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화 ID를 기준으로 유사한 영화 목록을 반환합니다.",
            "parameters": {
                "type": "object", 
                "properties": {
                    "id" : {
                        "type": "integer",
                        "description": "기준이 되는 영화의 ID (예: 693134)"
                    }, 
                },
            },
            "required": ["id"]
        }
    },
]


In [40]:
from openai.types.chat import ChatCompletion, ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage): 
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    } for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            
            print(f"Ran {function_name} with arg {arguments} for a result of {result}")

            messages.append(
                {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")
        
def call_ai():
    response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools = TOOLS,
    )
    process_ai_response(response.choices[0].message)



In [41]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message =="q":
        break
    else:
        messages.append({
            "role":"user",
            "content": message})
        print(f"User: {message}")
        call_ai()


User: Hi, my name is Kay
AI: Hi Kay! How can I assist you today?
User: Would you let me know popular movies?
Calling function: get_popular_movies with {}
Ran get_popular_movies with arg {} for a result of 1. Your Heart Will Be Broken (ID:1523145), 2. Avatar: Fire and Ash (ID:83533), 3. Hoppers (ID:1327819), 4. The Super Mario Bros. Movie (ID:502356), 5. The Super Mario Galaxy Movie (ID:1226863), 6. Shelter (ID:1290821), 7. Project Hail Mary (ID:687163), 8. Thrash (ID:1290417), 9. Crime 101 (ID:1171145), 10. Greenland 2: Migration (ID:840464), 11. GOAT (ID:1297842), 12. Mike & Nick & Nick & Alice (ID:1115544), 13. Sniper: No Nation (ID:1641319), 14. The Bride! (ID:1159831), 15. War Machine (ID:1265609), 16. The Mortuary Assistant (ID:1470130), 17. Pretty Lethal (ID:1084187), 18. Humint (ID:1268127), 19. Scream 7 (ID:1159559), 20. The Passion of the Christ (ID:615)
AI: Here are some popular movies:

1. **Your Heart Will Be Broken** (ID: 1523145)
2. **Avatar: Fire and Ash** (ID: 83533)
3.